# Random Forest sobre `data_model_borda30`

En esta notebook cargamos el dataset `data_model_borda30`, hacemos una división de datos segura para evitar fuga de datos, optimizamos los hiperparámetros de un modelo Random Forest y evaluamos el modelo usando **Weighted F1 Score** y **AUC-ROC**.

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score, confusion_matrix

random_state = 42
workspace_dir = r'c:\Users\ricar\OneDrive\Desktop\TESIS'
file_path = os.path.join(workspace_dir, 'data_model_borda30.csv')

if not os.path.exists(file_path):
    raise FileNotFoundError(f'No se encontró el archivo: {file_path}')

print('Cargando dataset:', file_path)
data = pd.read_csv(file_path)
print('Shape:', data.shape)
print(data.head())
print('\nColumnas:', data.columns.tolist())
print('\nTipo de label:', data['Type'].dtype)
print('\nDistribución de clases:')
print(data['Type'].value_counts())

In [ ]:
label_col = 'Type'
feature_cols = [c for c in data.columns if c != label_col]
X = data[feature_cols].copy()
y = data[label_col].copy()

if X.isna().any().any():
    X = X.fillna(X.median())
    print('Se imputaron valores nulos con la mediana.')

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print('Clases codificadas:', list(le.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=random_state
)

print('\nTamaño de split:')
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train clase counts:\n', pd.Series(y_train).value_counts(normalize=True))
print('y_test clase counts:\n', pd.Series(y_test).value_counts(normalize=True))

In [ ]:
# Random Forest no necesita escalado estricto, pero usamos pipeline en caso de extender con otros estimadores.
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=random_state, n_jobs=-1))
])

param_distributions = {
    'rf__n_estimators': [100, 200, 400, 800],
    'rf__max_depth': [None, 10, 20, 30, 50],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', 0.5],
    'rf__class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring='f1_weighted',
    n_jobs=-1,
    cv=cv,
    verbose=2,
    random_state=random_state,
    return_train_score=True
)

search.fit(X_train, y_train)

print('\nMejores hiperparámetros:')
print(search.best_params_)
print('\nMejor score CV (f1_weighted):', search.best_score_)

In [ ]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)

f1_weighted = f1_score(y_test, y_pred, average='weighted')
accuracy = accuracy_score(y_test, y_pred)
print('Test accuracy:', accuracy)
print('Test weighted F1:', f1_weighted)

n_classes = len(le.classes_)
if n_classes == 2:
    auc_roc = roc_auc_score(y_test, y_pred_proba[:, 1])
else:
    y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))
    auc_roc = roc_auc_score(y_test_bin, y_pred_proba, average='weighted', multi_class='ovr')

print('Test AUC-ROC (weighted):', auc_roc)

print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print('Matriz de confusión:')
print(cm)

In [ ]:
# Guardar modelo final entrenado y resultados adicionales
import joblib

model_path = os.path.join(workspace_dir, 'random_forest_borda30_best.pkl')
joblib.dump(best_model, model_path)
print('Modelo guardado en:', model_path)

results_path = os.path.join(workspace_dir, 'random_forest_borda30_results.txt')
with open(results_path, 'w', encoding='utf-8') as f:
    f.write('Best params:\n')
    f.write(str(search.best_params_) + '\n\n')
    f.write(f'CV f1_weighted: {search.best_score_}\n')
    f.write(f'Test accuracy: {accuracy}\n')
    f.write(f'Test weighted F1: {f1_weighted}\n')
    f.write(f'Test AUC-ROC: {auc_roc}\n\n')
    f.write('Classification report:\n')
    f.write(classification_report(y_test, y_pred, target_names=le.classes_))
    f.write('\nConfusion matrix:\n')
    f.write(np.array2string(cm))

print('Resultados guardados en:', results_path)